# Ogefmeeting — Test génération compte rendu (GPT)

**Circuit identique à l'application :**
1. **STT live (Deepgram)** pendant la réunion → texte final sauvegardé
2. **GPT** après la réunion → **rapport complet** (≈ 1 page minimum selon la durée)

**Structure du rapport (3 niveaux au choix) :**
- **Simple** — synthèse courte
- **Détaillé** — standard (recommandé)
- **Très détaillé** — exhaustif

**Plan :** Introduction → points ODJ avec **sous-points** (un par projet/dossier cité) → Conclusion.
Pas de sections globales « Décisions » / « Actions ».

> Noyau Jupyter : `Python (ogefmeeting-IA)`

## 1. Configuration

In [1]:
import json
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

from ogefrem_context import (
    ContexteReunion,
    DIRECTIONS_OGEFREM,
    LIBELLES_NIVEAU_DETAIL,
    NIVEAUX_DETAIL_CR,
    OGEFREM_PRESENTATION,
    construire_prompt_systeme,
    construire_prompt_utilisateur,
    max_tokens_pour_niveau,
    parser_reponse_json_brute,
)

IA_DIR = Path.cwd()
load_dotenv(IA_DIR / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini").strip() or "gpt-4o-mini"

assert OPENAI_API_KEY, "Définissez OPENAI_API_KEY dans IA/.env"
assert ".venv" in sys.executable.replace("\\", "/").lower(), (
    "Mauvais noyau — choisissez Python (ogefmeeting-IA)"
)

client = OpenAI(api_key=OPENAI_API_KEY)
SAMPLES = IA_DIR / "samples"
SAMPLES.mkdir(exist_ok=True)


def generer_brouillon_cr(
    reunion: ContexteReunion,
    transcription: str,
    niveau: str = "detaille",
) -> dict:
    """Texte STT final + métadonnées réunion → rapport CR JSON (par ODJ + sous-points)."""
    temp = 0.25 if niveau == "tres_detaille" else 0.35
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        temperature=temp,
        max_tokens=max_tokens_pour_niveau(niveau),
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": construire_prompt_systeme()},
            {
                "role": "user",
                "content": construire_prompt_utilisateur(reunion, transcription, niveau),
            },
        ],
    )
    contenu = resp.choices[0].message.content or "{}"
    return parser_reponse_json_brute(contenu)


print("Modèle OpenAI:", OPENAI_MODEL)
print("Directions chargées:", len(DIRECTIONS_OGEFREM))

Modèle OpenAI: gpt-4o-mini
Directions chargées: 16


## 2. Contexte OGEFREM (aperçu)

Ces informations sont injectées dans le **prompt système** GPT.

In [4]:
print(OGEFREM_PRESENTATION)
print("\n--- Directions ---\n")
for d in DIRECTIONS_OGEFREM:
    print(f"{d['code']:6} | {d['nom']}")

L'OGEFREM (Office de Gestion du Fret Multimodal) est un établissement public de la RDC
chargé de la régulation, du contrôle et de la facilitation du fret maritime et multimodal.
Il délivre et supervise des instruments de traçabilité (FERI, AD, FERE), accompagne les
opérateurs du fret, et coordonne les directions techniques, commerciales, financières
et de contrôle interne.

--- Directions ---

DG     | Direction Générale
DFM    | Direction du Fret Maritime
DTFM   | Direction du Transit et du Fret Multimodal
DFAC   | Direction des Facilitations et Affaires Commerciales
DGIT   | Direction de Gestion des Instruments de Traçabilité
DEP    | Direction des Études et de la Planification
DANTIC | Direction de l'Application des NTIC
DSG    | Direction du Secrétariat Général
DOCG   | Direction de l'Organisation et du Contrôle de Gestion
DRH    | Direction des Ressources Humaines
DFIN   | Direction Financière
DAI    | Direction de l'Audit Interne
DAJ    | Direction des Affaires Juridiques
DII    

In [5]:
pip install ipywidgets


   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 695.8 kB/s eta 0:00:03
   --------- ------------------------------ 0.5/2.2 MB 695.8 kB/s eta 0:00:03
   --------- ------------------------------ 0.5/2.2 MB 695.8 kB/s eta 0:00:03
   -------------- ------------------------- 0.8/2.2 MB 569.9 kB/s eta 0:00:03
   -------------- ------------------------- 0.8/2.2 MB 569.9 kB/s eta 0:00:03
   ------------------ --------------------- 1.0/2.2 MB 573.2 kB/s eta 0:00:03
   ------------------ --------------------- 1.0/2.2 MB 573.2 kB/s eta 0:00:03
   ----------------------- ---------------- 1.3/2.2 MB 595.8 kB/s eta 0:00:02
   ----------------------- ---------------- 1.3/2.2 MB 595.8 kB/s eta 0:00:02
   -------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\DEBUZE DAVID\Documents\Ogefrem\ProjetReunion\Ogefmeeting\IA\.venv\Scripts\python.exe -m pip install --upgrade pip


## 3. Transcription STT (texte final)

**Simulez le texte sauvegardé après une réunion live** :
- choisissez un **exemple DANTIC** (transcriptions longues dans `samples/`)
- ou **collez** directement votre transcription (bouton « Sauver texte » dans l'app)

Puis cliquez **Générer le CR** — GPT analysera d'abord l'intitulé, puis rédigera le rapport.

In [6]:
import ipywidgets as widgets
from IPython.display import display, Markdown

# Scénarios DANTIC — chaque exemple a ses métadonnées + fichier STT simulé
SCENARIOS: dict[str, dict] = {
    "Ogefmeeting — STT live et CR IA (DANTIC/DGIT)": {
        "fichier": "transcription_exemple_dantic.txt",
        "reunion": ContexteReunion(
            titre="Point d'avancement Ogefmeeting — transcription live et génération CR IA",
            type_reunion="technique",
            lieu="Salle DANTIC — Kinshasa",
            date_reunion="2026-08-24",
            directions_codes=["DANTIC", "DGIT"],
            description="Suivi du projet applicatif de gestion des réunions OGEFREM.",
            participants=[
                "David Debuze (DANTIC)",
                "Marie Kabongo (DANTIC — infra)",
                "Jean-Pierre Mulumba (DANTIC — dev)",
                "Grace Tshimanga (DGIT)",
            ],
            points_ordre_jour=[
                "Bilan transcription Deepgram en français/anglais",
                "Archivage audio et texte après clôture",
                "Tests génération CR avec GPT",
                "Préparation démo direction",
            ],
        ),
    },
    "Atelier technique FERI — API DANTIC/DGIT": {
        "fichier": "transcription_dantic_feri_dgit.txt",
        "reunion": ContexteReunion(
            titre="Modernisation des échanges de données autour du FERI",
            type_reunion="technique",
            lieu="Salle de conférence DANTIC — Kinshasa",
            date_reunion="2026-08-20",
            directions_codes=["DANTIC", "DGIT", "DRCP"],
            description="Atelier de cadrage technique sur les flux FERI et réduction des ressaisies.",
            participants=[
                "Patrick Ilunga (DGIT)",
                "Sandra Mbuyi (DGIT — FERI)",
                "Eric Nsimba (DANTIC — architecture)",
                "Clarisse Banza (DANTIC — analyse métier)",
            ],
            points_ordre_jour=[
                "État des flux actuels et irritants chargeurs",
                "Proposition API REST et modèle événementiel",
                "Sécurité, idempotence et environnements",
                "Planning et comité de pilotage",
            ],
        ),
    },
    "Comité cybersécurité — infrastructure DANTIC": {
        "fichier": "transcription_dantic_securite_infra.txt",
        "reunion": ContexteReunion(
            titre="Revue trimestrielle cybersécurité et continuité de service des applications OGEFREM",
            type_reunion="technique",
            lieu="Salle comité DANTIC — Kinshasa",
            date_reunion="2026-08-14",
            directions_codes=["DANTIC", "DAI"],
            description="Point sécurité, sauvegardes, accès et validation des services cloud pilotes.",
            participants=[
                "Joseph Kabeya (DANTIC)",
                "Léa Mutombo (RSSI)",
                "Paul Omekenge (admin systèmes)",
                "Nathalie Kasa (DAI)",
                "David Debuze (Ogefmeeting)",
            ],
            points_ordre_jour=[
                "Vulnérabilités et correctifs",
                "Sauvegardes et tests de restauration",
                "Données audio/transcription et conformité",
                "Validation SaaS Deepgram/OpenAI en pilote",
            ],
        ),
    },
}

# Variables partagées avec la cellule de génération
reunion: ContexteReunion | None = None
transcription: str = ""
brouillon_cr: dict = {}

selecteur_niveau = widgets.Dropdown(
    options=[(LIBELLES_NIVEAU_DETAIL[n], n) for n in NIVEAUX_DETAIL_CR],
    value="detaille",
    description="Niveau :",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="98%"),
)

selecteur = widgets.Dropdown(
    options=list(SCENARIOS.keys()),
    description="Exemple :",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="98%"),
)

zone_transcription = widgets.Textarea(
    value="",
    placeholder="Collez ici la transcription STT finale (texte sauvegardé après la réunion live)...",
    rows=22,
    layout=widgets.Layout(width="98%", height="420px"),
)

bouton_charger = widgets.Button(description="Charger l'exemple", button_style="info", icon="folder-open")
bouton_generer = widgets.Button(description="Générer le CR", button_style="success", icon="magic")
bouton_effacer = widgets.Button(description="Effacer le texte", icon="eraser")

zone_sortie = widgets.Output()


def _lire_fichier_exemple(nom_fichier: str) -> str:
    chemin = SAMPLES / nom_fichier
    if not chemin.exists():
        return f"[Fichier introuvable : {chemin}]"
    return chemin.read_text(encoding="utf-8")


def charger_exemple(_=None) -> None:
    global reunion, transcription
    scenario = SCENARIOS[selecteur.value]
    reunion = scenario["reunion"]
    transcription = _lire_fichier_exemple(scenario["fichier"])
    zone_transcription.value = transcription
    with zone_sortie:
        zone_sortie.clear_output()
        print(f"Exemple chargé : {selecteur.value}")
        print(f"Réunion : {reunion.titre}")
        print(f"Transcription : {len(transcription):,} caractères (~{len(transcription.split())} mots)")


def effacer_texte(_=None) -> None:
    zone_transcription.value = ""


def generer_cr(_=None) -> None:
    global reunion, transcription, brouillon_cr

    transcription = zone_transcription.value.strip()
    if not transcription:
        with zone_sortie:
            zone_sortie.clear_output()
            print("Collez une transcription ou chargez un exemple avant de générer.")
        return

    if reunion is None:
        reunion = SCENARIOS[selecteur.value]["reunion"]

    with zone_sortie:
        zone_sortie.clear_output(wait=True)
        print("Appel GPT en cours… (STT → compte rendu)")
        print(f"Modèle : {OPENAI_MODEL}")
        print(f"Niveau : {LIBELLES_NIVEAU_DETAIL[selecteur_niveau.value]}")
        print(f"Réunion : {reunion.titre}\n")

        brouillon_cr = generer_brouillon_cr(reunion, transcription, selecteur_niveau.value)

        chemin_sortie = SAMPLES / "brouillon_cr_exemple.json"
        chemin_sortie.write_text(
            json.dumps(brouillon_cr, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

        display(Markdown("### Introduction"))
        print(brouillon_cr.get("introduction", ""))
        display(Markdown("### Points d'ordre du jour"))
        for i, point in enumerate(brouillon_cr.get("points_ordre_jour") or [], 1):
            print(f"\n--- {i}. {point.get('titre', '')} ---")
            if point.get("contenu"):
                print(point.get("contenu"))
            for j, sp in enumerate(point.get("sous_points") or [], 1):
                print(f"\n  {i}.{j} {sp.get('titre', '')}")
                print(f"  {sp.get('contenu', '')}")
        display(Markdown("### Conclusion"))
        print(brouillon_cr.get("conclusion", ""))
        display(Markdown("### Directions impliquées"))
        print(", ".join(brouillon_cr.get("directions_impliquees", [])))
        print(f"\nJSON complet sauvegardé : {chemin_sortie}")


bouton_charger.on_click(charger_exemple)
bouton_generer.on_click(generer_cr)
bouton_effacer.on_click(effacer_texte)

display(
    widgets.VBox([
        selecteur,
        selecteur_niveau,
        widgets.HBox([bouton_charger, bouton_generer, bouton_effacer]),
        zone_transcription,
        zone_sortie,
    ])
)

# Charge le premier exemple au démarrage
charger_exemple()

## 4. Résultat JSON complet

Après génération, exécutez la cellule suivante pour voir tout le JSON.

In [ ]:
*(La génération se fait via le bouton **Générer le CR** dans la section 3.)*

## 5. Détail complet (JSON)

Exécutez cette cellule après avoir généré un CR (bouton ci-dessus).

In [7]:
if not brouillon_cr:
    print("Aucun brouillon — cliquez d'abord sur « Générer le CR » dans la cellule 3.")
else:
    print(json.dumps(brouillon_cr, ensure_ascii=False, indent=2))

{
  "analyse_intitule": "La réunion intitulée 'Point d'avancement Ogefmeeting' vise à faire le point sur le projet applicatif de gestion des réunions OGEFREM. La transcription aborde les avancées techniques, les tests effectués et les décisions prises concernant la mise en œuvre de l'application.",
  "directions_impliquees": [
    "DANTIC",
    "DGIT"
  ],
  "synthese": "La réunion a permis de faire un état des lieux sur le développement de l'application Ogefmeeting, qui inclut des fonctionnalités de transcription en temps réel et de génération automatique de comptes rendus. Les participants ont discuté des aspects techniques, des tests en cours et des prochaines étapes pour la démo à la direction.",
  "contexte": "Cette réunion technique s'inscrit dans le cadre du suivi du projet Ogefmeeting, qui a pour but d'améliorer la gestion des réunions au sein de l'OGEFREM grâce à des outils numériques avancés.",
  "points_traites": [
    {
      "titre": "Bilan transcription Deepgram",
      "

## 6. Intégration app (prochaine étape)

Si le résultat convient :
- réutiliser `ogefrem_context.py` côté backend (`cr-ia.service.ts` équivalent Python → port TS)
- route `POST /api/comptes-rendus/:id/generer-ia`
- bouton dans l'éditeur CR après clôture
- entrée = transcription sauvegardée + métadonnées réunion